# Chapter 03-04 · Probability, by counting

**Label:** Core  |  **Time:** ~50 minutes  |  **Difficulty:** gentle arithmetic, one genuinely
counter-intuitive result

**Prerequisites:** 03-01. Nothing from 03-02 or 03-03 is needed, though they make the last section
land harder.

**Position in the learning path:** module 03, chapter 4 of 8. Before: **03-03**. After: **03-05**,
Bayes' rule - which is this chapter's arithmetic written as a formula.

---

## Why this matters

Every probability in this chapter is a count divided by another count. No axioms, no distributions,
no integrals - a table of a thousand bikes and some division you can do on paper.

That is not a simplification for beginners. It is where the two most expensive mistakes in applied
probability become obvious:

- **Reading a conditional probability backwards.** "The detector catches 90% of faults" and "90% of
  alarms are faults" sound like the same sentence. Below, one is 0.90 and the other is 0.27.
- **Assuming independence.** Two components each fail 1% of the time. If you multiply, both failing
  is one chance in ten thousand. The real answer, from a mechanism that leaves each component's 1%
  exactly unchanged, is one in 247.

Both are counting errors. Both are easier to see in a table than in a formula, which is why this
chapter builds the table first and the notation second.

## What you will be able to do

- Read joint, marginal and conditional probabilities off a table of counts
- Say why `P(A given B)` and `P(B given A)` differ, and by how much
- Test whether two things are independent, and say what independence means in counts
- Explain why multiplying failure rates is usually optimistic, and by how much
- Recognise the same distinction when it reappears as precision and recall in module 07

## Warm-up: retrieve, do not reread

1. What does the bootstrap estimate, and what can it never detect?
2. What did a "95%" interval actually deliver at n = 7?
3. Why does the bootstrap fail completely for a maximum?

<br>

*Answers: (1) the sampling distribution - spread, never bias. (2) 0.859. (3) a resample cannot
contain a value the sample did not have, so every interval sits below the true maximum.*

## A thousand bikes

The workshop fits every bike with a sensor that raises an alarm when it detects a cracked frame. To
find out whether the sensor is any good, a thousand bikes are inspected by hand and the results
cross-tabulated against the alarm.

**Four numbers. Everything in this chapter comes out of them.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# SYNTHETIC but hand-checkable: 1,000 inspected bikes, whole numbers throughout.
counts = pd.DataFrame(
    {"alarm": [36, 96], "no alarm": [4, 864]},
    index=["cracked", "sound"],
)
counts.index.name = "reality"

total = int(counts.values.sum())
print(counts.to_string())
print("\ntotal bikes:", total)

### Predict before running

The sensor **catches 36 of the 40 cracked frames**, which is a hit rate of 90%. It raises a false
alarm on 96 of the 960 sound ones, which is 10%. By the standards of physical inspection equipment
that is a good sensor.

The workshop's actual question is different: **a bike's alarm has just gone off. What is the chance
its frame is cracked?**

Write a number down before running the next cell. Most people say something near 90%.

In [ ]:
cracked_and_alarm = counts.loc["cracked", "alarm"]
all_alarms = counts["alarm"].sum()
all_cracked = counts.loc["cracked"].sum()

print("P(alarm | cracked) = %d / %d = %.4f   <- the sensor's hit rate"
      % (cracked_and_alarm, all_cracked, cracked_and_alarm / all_cracked))
print("P(cracked | alarm) = %d / %d = %.4f   <- what the workshop actually needs"
      % (cracked_and_alarm, all_alarms, cracked_and_alarm / all_alarms))
print()
print("the two differ by a factor of %.2f" % ((cracked_and_alarm / all_cracked)
                                              / (cracked_and_alarm / all_alarms)))

## Failure lab: the same 36 bikes, two different denominators

**0.90 and 0.27 are computed from the same numerator.** The only thing that changed is what you
divided by.

- `P(alarm | cracked)` restricts attention to the **40 cracked bikes** and asks how many alarmed: 36.
- `P(cracked | alarm)` restricts attention to the **132 bikes that alarmed** and asks how many are
  cracked: 36.

Those two groups are different sizes, so the answers differ. **Nearly three quarters of alarms are on
sound bikes**, and that is entirely compatible with a sensor that catches 90% of cracks.

### Why: there are far more sound bikes to be wrong about

The sensor is wrong about 10% of sound frames, and there are 960 of them - so it produces 96 false
alarms. It is wrong about 10% of cracked frames, and there are only 40 - so it misses 4. The false
alarms outnumber the true ones **96 to 36**, not because the sensor is bad but because cracked frames
are rare.

**This is the base rate**, and it is what makes the reversal counter-intuitive. Change nothing about
the sensor and make cracks common instead of rare, and the same 90% hit rate produces a completely
different answer to the workshop's question.

In [ ]:
def sensor_table(rate_of_cracks, hit_rate=0.90, false_alarm_rate=0.10, fleet=1000):
    cracked = fleet * rate_of_cracks
    sound = fleet - cracked
    true_alarms = cracked * hit_rate
    false_alarms = sound * false_alarm_rate
    return true_alarms / (true_alarms + false_alarms)


rows = []
for rate in [0.001, 0.01, 0.04, 0.10, 0.30, 0.60]:
    rows.append({"share of bikes cracked": "%.1f%%" % (100 * rate),
                 "P(alarm | cracked)": 0.90,
                 "P(cracked | alarm)": round(sensor_table(rate), 4)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nthe sensor is identical in every row.")

The sensor never changes. `P(cracked | alarm)` runs from **0.009 to 0.931** purely because of how
common cracks are.

**This is why the same detector is excellent in one setting and useless in another**, and it is the
single most important idea in the chapter:

| Setting | Base rate | What an alarm means |
|---|---|---|
| Screening a whole fleet | 0.1% | 99% of alarms are false. The sensor is a nuisance |
| Bikes already flagged by a mechanic | 30% | Most alarms are real. The sensor is useful |

Same equipment, opposite verdicts. **A detector's usefulness is not a property of the detector.** It
depends on what you point it at - which is why a model validated on a balanced dataset and deployed
on a rare event disappoints, and why module 06 spends a chapter on imbalance.

### You have already met these two numbers under other names

| This chapter | Module 07 calls it | In words |
|---|---|---|
| `P(alarm given cracked)` = 0.90 | **recall** | of the things that are real, how many did we catch |
| `P(cracked given alarm)` = 0.27 | **precision** | of the things we flagged, how many are real |

Precision and recall are not two arbitrary metrics to memorise. They are **the same fraction with the
two different denominators from this table**, and the reason both are always reported is that neither
one alone tells you what an alarm means.

## The definition, now that you have used it

Every probability so far has been *"count the rows you care about, divide by the rows you are
considering"*. Written out:

- **Marginal** - `P(cracked)` = 40 / 1000 = 0.040. One row's total over everything.
- **Joint** - `P(cracked and alarm)` = 36 / 1000 = 0.036. One cell over everything.
- **Conditional** - `P(cracked | alarm)` = 36 / 132 = 0.273. One cell over its column's total.

And the relationship people memorise as a formula is just those three lines divided by each other:

> `P(A | B) = P(A and B) / P(B)`

Check it against the counts: `0.036 / 0.132 = 0.273`. The formula is the table.

In [ ]:
p_cracked = all_cracked / total
p_alarm = all_alarms / total
p_both = cracked_and_alarm / total

print("P(cracked)            = %.4f" % p_cracked)
print("P(alarm)              = %.4f" % p_alarm)
print("P(cracked and alarm)  = %.4f" % p_both)
print()
print("P(cracked | alarm) via the formula : %.4f / %.4f = %.4f" % (p_both, p_alarm, p_both / p_alarm))
print("P(cracked | alarm) by counting     : %d / %d = %.4f"
      % (cracked_and_alarm, all_alarms, cracked_and_alarm / all_alarms))

## Independence, in counts

Two things are **independent** when knowing one tells you nothing about the other:

> `P(A | B) = P(A)`

The alarm and the crack are obviously not independent - knowing a bike alarmed moves your estimate
from 4% to 27%, which is the whole point of having a sensor.

Here is a variable that is. The workshop also recorded frame colour.

In [ ]:
colour = pd.DataFrame(
    {"cracked": [16, 24], "sound": [384, 576]},
    index=["red", "black"],
)
colour.index.name = "frame colour"
print(colour.to_string())
print()
for shade in ["red", "black"]:
    cracked_here = colour.loc[shade, "cracked"]
    total_here = colour.loc[shade].sum()
    print("P(cracked | %-5s) = %3d / %3d = %.4f" % (shade, cracked_here, total_here,
                                                   cracked_here / total_here))
print("P(cracked)          = %3d / %d = %.4f" % (all_cracked, total, p_cracked))

Knowing the colour changes nothing: **0.0400 either way, and 0.0400 overall.** Colour and cracking
are independent.

### The other way of checking, which is the one that generalises

If two things are independent, the count in each cell should be **the row total times the column
total, divided by the grand total** - what you would get if the two classifications were assigned
without reference to each other.

In [ ]:
def expected_counts(table):
    row_totals = table.sum(axis=1).to_numpy()[:, None]
    column_totals = table.sum(axis=0).to_numpy()[None, :]
    return pd.DataFrame(row_totals * column_totals / table.values.sum(),
                        index=table.index, columns=table.columns)


print("COLOUR - observed")
print(colour.to_string())
print("\nCOLOUR - expected if independent")
print(expected_counts(colour).to_string())

print("\n\nSENSOR - observed")
print(counts.to_string())
print("\nSENSOR - expected if independent")
print(expected_counts(counts).round(1).to_string())

For colour, expected and observed are **identical** - 16, 384, 24, 576. Independence, exactly.

For the sensor, independence would predict **5.3 cracked-and-alarmed bikes**, and there are **36**.
Nearly seven times as many, which is the sensor working.

That gap - observed against expected-under-independence - is the basis of every test of association
between two categorical variables, including the chi-squared test. You now know what such a test is
looking at, which is more useful than knowing its formula.

## Failure lab 2: two components, each 1% unreliable

A bike has two brake cables. Each fails on about **1% of bikes**. The brakes fail completely only if
**both** cables fail.

The standard calculation multiplies: `0.01 x 0.01 = 0.0001`, one bike in ten thousand. On a fleet of
five thousand that is half a bike, so nobody worries.

Cables come in batches. Occasionally a batch is bad.

**Predict before running:** the numbers below are chosen so that each cable still fails on almost
exactly 1% of bikes - the individual failure rate you would measure in any test is unchanged. What
happens to the chance that *both* fail?

In [ ]:
bad_batch_rate = 0.02        # 2% of batches are bad
fail_if_bad = 0.45           # a cable from a bad batch fails 45% of the time
fail_if_good = 0.001         # a cable from a good batch fails 0.1% of the time

one_fails = bad_batch_rate * fail_if_bad + (1 - bad_batch_rate) * fail_if_good
both_fail = bad_batch_rate * fail_if_bad ** 2 + (1 - bad_batch_rate) * fail_if_good ** 2

print("P(a given cable fails)        = %.5f   (%.3f%%)" % (one_fails, 100 * one_fails))
print()
print("P(both fail), if independent  = %.8f   1 in %6.0f" % (one_fails ** 2, 1 / one_fails ** 2))
print("P(both fail), actual          = %.8f   1 in %6.0f" % (both_fail, 1 / both_fail))
print()
print("the real risk is %.1f times the independent calculation" % (both_fail / one_fails ** 2))

In [ ]:
# Confirm by simulating ten million bikes rather than trusting the algebra.
sim = np.random.default_rng(0)
n_bikes = 10_000_000

from_bad_batch = sim.random(n_bikes) < bad_batch_rate
failure_chance = np.where(from_bad_batch, fail_if_bad, fail_if_good)
first_fails = sim.random(n_bikes) < failure_chance
second_fails = sim.random(n_bikes) < failure_chance

print("simulated P(a cable fails) : %.5f" % first_fails.mean())
print("simulated P(both fail)     : %.6f   1 in %.0f"
      % ((first_fails & second_fails).mean(), 1 / (first_fails & second_fails).mean()))
print("simulated ratio to the independent calculation: %.1f"
      % ((first_fails & second_fails).mean() / (first_fails.mean() * second_fails.mean())))
print()
print("P(second fails | first failed) = %.4f    against P(fails) = %.5f"
      % ((first_fails & second_fails).mean() / first_fails.mean(), first_fails.mean()))

### Diagnosis: the marginal rates were never the problem

Each cable fails on **0.998%** of bikes - so a test of individual cables, however large, returns "1%"
and never hints at anything wrong.

But both fail on **1 bike in 247**, not 1 in 10,040. **Forty times the risk**, from an assumption
rather than from a measurement.

The mechanism is visible in the last line: once you know the first cable failed, the chance the
second fails is **0.4082** rather than 0.00997. The first failure is evidence that this bike came
from a bad batch, and everything from a bad batch is at risk together.

**Where this bites, in roughly increasing order of cost:**

- Two servers in the same rack, on the same power supply.
- Two models in an ensemble, trained on the same data with the same bug.
- Two features in a dataset, both derived from the same broken sensor.
- Two safeguards in a system, both written by the same team on the same assumptions.
- Mortgage defaults across a region, priced as if each household were independent.

In every case the individual rates are measured correctly and the joint rate is calculated by
multiplying. **Multiplying probabilities is a claim about mechanism**, not a step in arithmetic, and
the claim is that nothing connects the two events. It is the most common unexamined assumption in
applied work, and it always errs in the direction of optimism, because shared causes create failures
that arrive together.

### The rule that is always true

`P(A and B) = P(A) x P(B | A)`

That version needs no assumption. `P(A) x P(B)` is the special case where `P(B | A) = P(B)` - and
whether that holds is a question about the world, to be argued for rather than assumed.

## Common misconceptions

**"P(A given B) and P(B given A) are close enough."**
0.90 and 0.27, from the same 36 bikes. They are equal only when `P(A)` and `P(B)` are equal, which
is rare enough to be worth checking every time.

**"A 90% accurate test means a positive result is 90% likely to be right."**
It means whatever the base rate makes it mean - 0.9% at a base rate of 1 in 1,000, and 93% at 60%.

**"The sensor is good, so its alarms are trustworthy."**
Those are different claims. A good sensor pointed at a rare event produces mostly false alarms, and
no improvement in the sensor changes that as much as changing what you point it at.

**"Independent means unrelated in a vague sense."**
It has an exact meaning in counts: `P(A | B) = P(A)`, equivalently every cell equals row total times
column total over grand total. It is checkable, and worth checking.

**"Multiplying probabilities is how you combine them."**
Multiplying is how you combine *independent* probabilities. The general rule is
`P(A and B) = P(A) x P(B | A)`, and using the shortcut is an assertion that nothing links the two.

**"Rare events are safe to ignore because the numbers are small."**
The cable calculation moved from 1 in 10,040 to 1 in 247 without changing a single measured rate. On
a fleet of 5,000 that is the difference between half a bike and twenty.

**"If two variables are independent overall, they are independent within subgroups."**
False, and it is 02-06's Simpson's paradox in probability notation. Exercise E9.

## Exercises

Solutions: `solutions/03_math_foundations/03-04_probability_solutions.ipynb`.

### Quick understanding

**E1.** In one sentence each, describe what `P(alarm | cracked)` and `P(cracked | alarm)` count -
naming the denominator of each.

**E2.** Why does a good detector produce mostly false alarms when it is used to screen for a rare
condition?

**E3.** State what independence means in terms of counts in a table, without using the word
probability.

### Hand calculation

**E4.** From the sensor table, compute on paper: `P(sound | no alarm)`, `P(no alarm | sound)` and
`P(sound)`. Which two are close, and why is the third different?

**E5.** A second workshop inspects 500 bikes with the same sensor and finds 100 cracked frames.
Assuming the same 90% hit rate and 10% false-alarm rate, build the four-cell table and compute
`P(cracked | alarm)`. Compare with 0.273 and explain the whole difference in one sentence.

**E6.** Three cables now, all from the same batch, with the batch mechanism from the chapter. Compute
`P(all three fail)` under independence and under the batch model. What is the ratio, and how does it
compare with the two-cable ratio of 40.7?

### Coding

**E7.** Write `table_probabilities(table)` that prints every marginal, joint and conditional
probability derivable from a two-by-two table, labelled. Run it on the sensor table and check the
numbers against the chapter.

**E8.** Write `independence_gap(table)` returning observed minus expected counts, and a single number
summarising the discrepancy. Run it on the colour table and on the sensor table.

**E9.** Build a three-variable example where two variables are **independent overall but dependent
within each level of a third**. Verify both claims with counts. (Hint: the structure of 02-06 works -
choose the group sizes so the pooled table comes out exactly independent.)

### Interpretation

**E10.** A screening programme tests 100,000 people for a condition affecting 1 in 2,000. The test
has a 99% hit rate and a 1% false-positive rate. Build the table, compute the chance a positive
result is real, and write the sentence you would put in the letter sent to people who test positive.

**E11.** A fraud model flags 2% of transactions and catches 80% of fraud. Fraud is 0.3% of
transactions. Compute precision and recall. The team wants to raise recall to 95%. Without doing the
arithmetic, say what that will do to precision and why.

### Debugging

**E12.** A reliability report states: "each of our three data centres has 99.9% uptime, so the chance
all three are down simultaneously is one in a billion." Name the assumption, give two concrete
reasons it may fail here, and say which direction the estimate is wrong in.

### Exam and interview reasoning

**E13.** "Explain the difference between precision and recall to a product manager, and say when you
would optimise for each." Five sentences, using the sensor table rather than definitions.

### Transfer to a different situation

**E14.** An email filter marks 4% of mail as spam and is right 95% of the time when it does. You
receive 200 messages a day, of which 6% are genuinely spam. How many legitimate emails go to the
spam folder each day, and how many spam messages reach the inbox? Which error would you rather have,
and what does that imply about where to set the threshold?

### Explain it to someone non-technical

**E15.** A friend has tested positive for a rare condition on a "99% accurate" test and is
distraught. Explain, kindly and in under 120 words, why the number they should be thinking about is
not 99% - without telling them the test is unreliable, because it is not.

### Optional challenge

**E16.** The chapter's cable model has one hidden batch variable. Suppose you can only observe
failures, not batches. Given a fleet of 100,000 bikes with two cables each, can you detect that the
failures are dependent? Simulate it, propose a statistic, and say how large the fleet needs to be
before the dependence is unmistakable.

In [ ]:
# Your workspace. In memory: counts, colour, expected_counts, sensor_table,
# total, all_cracked, all_alarms, cracked_and_alarm.

## Mastery check

- [ ] Read joint, marginal and conditional probabilities off a two-by-two table without a formula
- [ ] Explain why `P(A | B)` and `P(B | A)` differ, using denominators
- [ ] Say how a base rate changes what a positive result means, with a number
- [ ] Check independence two ways: conditional equals marginal, and observed equals expected
- [ ] Explain why multiplying failure rates is an assumption about mechanism
- [ ] Connect `P(alarm | cracked)` and `P(cracked | alarm)` to recall and precision

## What should now feel instinctive

- Hearing "90% accurate" and immediately asking what the base rate is
- Drawing a two-by-two table of counts before reasoning about any conditional claim
- Treating every product of probabilities as a claim that needs defending
- Asking what the two things have in common before assuming they fail independently
- Noticing when a sentence has silently swapped its denominator

## Flashcards

| Front | Back |
|---|---|
| Conditional probability | Restrict to the rows where the condition holds, then count |
| `P(A given B)` in counts | cell / column total. `P(B given A)` is cell / row total |
| Sensor table | `P(alarm given cracked)` 0.90, `P(cracked given alarm)` 0.273, same 36 bikes |
| Why they differ | 960 sound bikes produce 96 false alarms; 40 cracked produce 36 true ones |
| Base rate effect | Same sensor gives 0.009 at a 0.1% rate and 0.931 at 60% |
| Recall and precision | `P(flag given real)` and `P(real given flag)` |
| Independence, in counts | every cell = row total x column total / grand total |
| The always-true rule | `P(A and B) = P(A) x P(B given A)` |
| Two 1% cables, shared batch | Both fail 1 in 247, not 1 in 10,040 - a factor of 40.7 |

## Next

**03-05 · Bayes' rule you can do on paper.** Everything needed for it is now in place: this chapter
computed `P(cracked | alarm)` from a table of counts, and Bayes' rule is that same calculation
written so it can be done when you have rates rather than counts - which is the situation you are
usually in.